# Linking the Layers of Your Data

Problem VIIIb of
[DSFP Session 27](https://github.com/LSSTC-DSFP/Session-27).

In lecture, I emphasized how images, catalogs, and light curves
comprise a single interlinked dataset.  I pointed out that tools which
previously would have required weeks to implement can now be prompted
to an LLM in minutes.  This is a surprise hack session intended to test that claim.

Your task is to **pick two layers of a dataset, and to prompt an LLM
to produce an interface that links them**.

This is **open-ended**: the point is for you to take this in a direction
that you want.  On Tuesday, almost everyone told me that they work with data,
and the exercise will work best if you bring something that _you_ have been
thinking about!

**After reading this page, close it.**  The work here happens in a terminal, an
IDE, a text editor, or a browser; not Jupyter!

* **11:00**: choose your data and your plan.  10 minutes.
* **11:10**: build, check, fix.  30 minutes.
* **11:40**: share.  15 minutes.

You are unlikely to finish a polished tool.  A rough prototype that answers one
question is the goal.  If it seems promising, consider it for this week's
hack!

* * *

By Luke Bouma (Caltech/IPAC)

## Your task, elaborated

1. **Link two layers**  Find two views of the same object(s),
   where moving between them is by default a challenge: a catalog and the
   light curves under it, a summary plot and the spectrum behind each
   point, an image and the table made from it.  Decide what a click
   or a selection in one view does to the other.
2. **One known answer, chosen before you build.**  An object you
   already understand, and what the tool must show for it.
3. **Prompt.**  Tell the LLM your question, where the data are and
   what the columns or arrays mean, your desired views, the link, and your
   known answer.  Ask for something you can open, for instance a
   script that writes a self-contained HTML page.  (As you might be able to
   tell, javascript can do a lot!)
4. **Check, fix, iterate.**  Run the tool on the known answer before you
   look at anything new.  Something will be wrong.  Iterate with the LLM.

No command line LLM on your laptop?  Pair with someone who has one.  Discuss
   your design plan and prompts together.

## A few datasets to poke at if you're stuck

Your own data is the best choice.  However, if you have none available, or just want to try something new, these
three are in **[this dropbox folder](https://www.dropbox.com/scl/fo/xplx0gcs47585q0j4i5vf/ADKcfZ1EAMlTB0bDf8H3_eM?rlkey=jlab3wt0bgp5dv0mupcd3dojv&st=q946cged&dl=0)**.  Each
subdirectory has a README; point your LLM at it.

| | Dataset | Layers | Download |
|---|---|---|---|
| 1 | Sco-Cen with TESS (Tuesday's sample) | light curves of 2287 young stars | 330 MB (or 44 MB for 300 stars) |
| 2 | SPHEREx spectra | ...for 10,000 nearby stars with Gaia and TESS properties; 4.8> flux measurements from 0.75 to 5 microns | 236 MB |
| 3 | Gaia DR3 activity | 297,000 stars within 200 pc: positions, motions, colors, chromospheric activity; cluster labels and rotation periods for some | 35 MB |

Some keys are shared, so you could join two of these as a project as well (for instance, on TIC identifier or Gaia source_id).

**1. Sco-Cen.**  `scocen/scocen_tess_2min.npz`, the file from
Tuesday.  The 300-star subsample is
**[here](https://www.dropbox.com/scl/fi/iyemr1up41snhzaj6vu7m/scocen_tess_2min_300.npz?rlkey=cd5lhbv9eo2lnoqx5izk6aptx&st=598cffoq&dl=1)**.

*Example*: measure a period and an amplitude for every star,
plot one against the other, and click a point to see that star's
light curve, folded and unfolded.
*Known truth*: a star whose period you measured on Tuesday.

**2. SPHEREx.**  `spherex_1e4/`: `stars.parquet` has one row per star,
`rows.parquet` has one row per star per image, and `README.txt`
defines every column.

*Example*: plot the stars in a color-magnitude diagram, or
rotation period against color, and click a star to see its spectrum.
Or link wavelength to time: does any star vary?
*Known truth*: the binned spectrum in `stars.parquet` (the
`spec_flux_NN` columns) must run through the same star's points from
`rows.parquet`.

**3. Gaia.**  `gaia_dr3_activity/`: `stars_200pc.csv.gz` and the
smaller `active_500pc.csv.gz`, with the same columns, defined in
`README.txt`.

*Example*: select a clump of stars in proper motion, or in
position, and see where they fall in the color-magnitude diagram and
in activity against temperature.  Which clumps are young?
*Known truth*: the Pleiades.  The clump at (`pmra`, `pmdec`) =
(+20, -45) mas/yr and 136 pc must fall on one narrow main sequence.

<details>
<summary>Click here to see how to read in the data.</summary>

**1. Sco-Cen**

```python
import numpy as np

f = np.load("scocen/scocen_tess_2min.npz")
ticid = f["ticid"]                       # (n_star,)
splits = np.cumsum(f["n_points"])[:-1]
times = np.split(f["time"], splits)      # one array per star, days
fluxes = np.split(f["flux"], splits)     # median-normalized
```

**2. SPHEREx**  (reading parquet needs `pyarrow`)

```python
import pandas as pd

stars = pd.read_parquet("spherex_1e4/stars.parquet")
rows = pd.read_parquet("spherex_1e4/rows.parquet")
one = rows[rows.ticid == stars.ticid.iloc[0]]
# one.wavelength_um, one.flux_jy, one.flux_err_jy, one.mjd_avg
```

**3. Gaia**

```python
import pandas as pd

df = pd.read_csv("gaia_dr3_activity/active_500pc.csv.gz")
# sky and space: ra, dec, x_pc, y_pc, z_pc
# motion: pmra, pmdec, v_l, v_b
# photometry: bp_rp, M_G
# activity: teff_gspphot, log10_rprime_irt, ew_espels_halpha
# sparse: cdips_cluster, cdips_log10_age, vrm_period
```

</details>

## Share

At 11:40, be ready to share what you did.  Can you see anything now that you couldn't before?  Do you have any successes or challenges you'd like to highlight?

Is there a problem in your own research that a
purpose-built tool like this would fix?  Consider it as a hack pitch for this afternoon!

## Notes for the debrief

There are no solutions!  However, here are a few areas we might wish to discuss.

1. What worked?  What didn't?
2. What verification strategies did you find effective?
3. Did you notice anything the LLMs persistently got wrong?

There are of course many other issues connected to the use of LLMs in astronomy.  While I do hope to focus today's session on using them to learn new things about our data, here is a **[google drive folder](https://docs.google.com/presentation/d/1T5VmfOBG8G7SrmyYUHK5k_ErUGkekbnCuPPexTKPgfw/edit?usp=sharing)** with some broader considerations, which I'm also happy to discuss.

## References

* Rebull et al. 2022, rotation in Upper Centaurus-Lupus and Lower
  Centaurus-Crux with TESS,
  [2022AJ....164...80R](https://scixplorer.org/abs/2022AJ....164...80R/abstract)
* Lanzafame et al. 2023, the Gaia DR3 Ca II infrared triplet activity
  index,
  [2023A&A...674A..30L](https://scixplorer.org/abs/2023A&A...674A..30L/abstract)
* Distefano et al. 2023, Gaia DR3 rotational modulation,
  [2023A&A...674A..20D](https://scixplorer.org/abs/2023A&A...674A..20D/abstract)